# 04 — Pseudo-Absence / Background Sampling

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

## Stage-3 result used by this notebook

The reviewed Malaysia positive inventory contains:

- **287 unique positive landslide records**
- 235 NASA COOLR
- 52 JMG
- 1 strong COOLR duplicate removed
- 0 JMG duplicate rows removed
- 0 strong JMG↔COOLR duplicate rows removed

This notebook creates defensible **pseudo-absence / background control points** for the first susceptibility model.

---

## Why pseudo-absence points are needed

The landslide inventories provide locations where landslides were observed, but they do not provide verified national-scale locations where landslides never occurred.

Therefore, target = 0 samples should be described as:

> **pseudo-absence / background controls**

rather than confirmed non-landslide observations.

---

## Sampling design

The sampling strategy is deliberately conservative:

1. Sample only inside the supplied Malaysia land boundary.
2. Exclude areas surrounding known positive landslides.
3. Use source location accuracy where available to avoid sampling too close to uncertain COOLR locations.
4. Use precise JMG points with a minimum exclusion distance.
5. Distribute pseudo-absences spatially using equal-area grid cells rather than simple national random sampling.
6. Enforce a minimum separation between pseudo-absence points.
7. Use a fixed random seed for reproducibility.
8. Create nested:
   - 1:1
   - 1:2 **baseline**
   - 1:3
   positive:pseudo-absence datasets.
9. Do not yet extract environmental predictors in this notebook.

### Recommended first modelling ratio

The primary first experiment will use:

**287 positives + 574 pseudo-absences = 861 samples**

This gives a 1:2 positive:pseudo-absence ratio while keeping the dataset manageable.

The 1:1 and 1:3 versions are saved for sensitivity analysis later.


In [27]:
# OPTIONAL: run only if packages are missing.
# !pip install -q geopandas pandas numpy matplotlib shapely pyproj

from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import Point, box
from shapely.ops import unary_union

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


pandas: 3.0.2
geopandas: 1.1.4


## 1. Project paths

Change **only `PROJECT_ROOT`**.

Expected input:

```text
Landslide_FYP_Development/
└── 02_cleaned_inventory/
    └── malaysia_positive_inventory_final.geojson
```

Output location:

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   └── pseudo_absence_generation_outputs/
└── 02_cleaned_inventory/
    ├── pseudo_absence_master_1to3.geojson
    ├── modelling_samples_1to1.geojson
    ├── modelling_samples_1to2.geojson
    └── modelling_samples_1to3.geojson
```


In [28]:
# ============================================================
# CHANGE ONLY THIS LINE
# ============================================================
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

POSITIVE_PATH = (
    DEV_DIR
    / "02_cleaned_inventory"
    / "malaysia_positive_inventory_final.geojson"
)

BOUNDARY_PATH = (
    RAW_DIR
    / "02_Boundary"
    / "mys_admin0.geojson"
)

OUTPUT_DIR = (
    DEV_DIR
    / "01_preprocessing"
    / "pseudo_absence_generation_outputs"
)

CLEAN_DIR = DEV_DIR / "02_cleaned_inventory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Final positive inventory": POSITIVE_PATH,
    "Malaysia boundary": BOUNDARY_PATH,
}.items():
    print(f"{name}: {path}")
    if not path.exists():
        raise FileNotFoundError(f"Missing required file:\\n{path}")

print("\\nRequired Stage-4 inputs found.")


Final positive inventory: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_final.geojson
Malaysia boundary: C:\Landslide_FYP\Landslide_FYP_Data\02_Boundary\mys_admin0.geojson
\nRequired Stage-4 inputs found.


## 2. Load and validate the final positive inventory


In [29]:
positive = gpd.read_file(POSITIVE_PATH)
malaysia = gpd.read_file(BOUNDARY_PATH)

print("Positive inventory:", positive.shape)
print("Positive CRS:", positive.crs)
print("Malaysia boundary:", malaysia.shape)
print("Boundary CRS:", malaysia.crs)

if len(positive) != 287:
    print(
        f"WARNING: Stage 3 previously produced 287 positives, "
        f"but this file currently contains {len(positive)}."
    )

if positive.crs is None or malaysia.crs is None:
    raise ValueError("Input CRS is missing.")

positive = positive.to_crs("EPSG:4326")
malaysia = malaysia.to_crs("EPSG:4326")

if "final_positive_id" not in positive.columns:
    raise KeyError("final_positive_id is missing from the Stage-3 output.")

if positive["final_positive_id"].duplicated().any():
    raise ValueError("Duplicate final_positive_id values found.")

print("\nPositive source counts:")
display(
    positive["source_dataset"]
    .value_counts()
    .rename_axis("source_dataset")
    .reset_index(name="records")
)


Positive inventory: (287, 20)
Positive CRS: EPSG:4326
Malaysia boundary: (1, 18)
Boundary CRS: EPSG:4326

Positive source counts:


,source_dataset,records
0,NASA_COOLR,235
1,JMG,52


## 3. Sampling parameters

### Equal-area CRS

All buffer distances and sampling calculations use **EPSG:6933** (World Equidistant Cylindrical / equal-area-style global metric workflow suitable for area/distance-based national sampling).

WGS84 (`EPSG:4326`) is used only for final storage/display.

### Default settings

- baseline ratio = **1:2**
- master pool = **1:3**
- grid size = **50 km**
- minimum pseudo-absence separation = **2 km**
- default minimum positive exclusion = **2 km**
- reproducible random seed = **42**

### Accuracy-aware exclusion

For NASA COOLR records, the supplied `location_accuracy_source` field is used when it can be interpreted safely.

Examples:

- `exact` → 2 km minimum exclusion
- `1km` → 2 km minimum exclusion
- `5km` → 5 km
- `10km` → 10 km
- `25km` → 25 km
- `50km` → 50 km

For precise JMG source geometry:

- 2 km minimum exclusion

This reduces the chance of assigning a pseudo-absence inside the positional uncertainty region of a known landslide.


In [30]:
RANDOM_SEED = 42

BASELINE_RATIO = 2
MASTER_RATIO = 3

GRID_SIZE_M = 50_000
MIN_PSEUDO_SEPARATION_M = 2_000
DEFAULT_POSITIVE_EXCLUSION_M = 2_000

SAMPLING_CRS = "EPSG:6933"

rng = np.random.default_rng(RANDOM_SEED)

N_POS = len(positive)
N_1TO1 = N_POS
N_1TO2 = N_POS * 2
N_1TO3 = N_POS * 3

print("Positive records:", N_POS)
print("1:1 pseudo-absences:", N_1TO1)
print("1:2 pseudo-absences:", N_1TO2)
print("1:3 pseudo-absences:", N_1TO3)
print("Baseline total samples:", N_POS + N_1TO2)


Positive records: 287
1:1 pseudo-absences: 287
1:2 pseudo-absences: 574
1:3 pseudo-absences: 861
Baseline total samples: 861


## 4. Create accuracy-aware positive exclusion distances

This step does **not** alter the positive locations.

It only determines how close a pseudo-absence may be sampled to each known landslide.


In [31]:
def parse_accuracy_to_m(value, default_m=DEFAULT_POSITIVE_EXCLUSION_M):
    if pd.isna(value):
        return float(default_m)

    text = str(value).strip().lower()

    if text in {"exact", "precise", "point"}:
        return float(default_m)

    # Find first numeric value.
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if not match:
        return float(default_m)

    number = float(match.group(1))

    if "km" in text:
        metres = number * 1000.0
    elif "m" in text:
        metres = number
    else:
        # COOLR accuracy labels are commonly km-oriented.
        # If unit is not explicit, do not assume a smaller distance
        # than the project minimum.
        metres = default_m

    return max(float(default_m), metres)


positive["exclusion_buffer_m"] = DEFAULT_POSITIVE_EXCLUSION_M

coolr_mask = positive["source_dataset"].eq("NASA_COOLR")

if "location_accuracy_source" in positive.columns:
    positive.loc[coolr_mask, "exclusion_buffer_m"] = (
        positive.loc[coolr_mask, "location_accuracy_source"]
        .apply(parse_accuracy_to_m)
    )

# JMG is based on original source point geometry.
jmg_mask = positive["source_dataset"].eq("JMG")
positive.loc[jmg_mask, "exclusion_buffer_m"] = DEFAULT_POSITIVE_EXCLUSION_M

buffer_summary = (
    positive.groupby(
        ["source_dataset", "exclusion_buffer_m"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values(["source_dataset", "exclusion_buffer_m"])
)

display(buffer_summary)

buffer_summary.to_csv(
    OUTPUT_DIR / "positive_exclusion_buffer_summary.csv",
    index=False
)


,source_dataset,exclusion_buffer_m,records
0,JMG,2000,52
1,NASA_COOLR,2000,88
2,NASA_COOLR,5000,66
3,NASA_COOLR,10000,31
4,NASA_COOLR,25000,34
5,NASA_COOLR,50000,15
6,NASA_COOLR,250000,1


## 5. Build the eligible pseudo-absence sampling area

The eligible area is:

> Malaysia land boundary − union of all positive exclusion buffers

No point is sampled from the excluded zones.


In [32]:
positive_m = positive.to_crs(SAMPLING_CRS)
malaysia_m = malaysia.to_crs(SAMPLING_CRS)

if len(malaysia_m) == 1:
    malaysia_geom_m = malaysia_m.geometry.iloc[0]
else:
    malaysia_geom_m = malaysia_m.dissolve().geometry.iloc[0]

positive_buffers = [
    geom.buffer(float(dist))
    for geom, dist in zip(
        positive_m.geometry,
        positive_m["exclusion_buffer_m"]
    )
]

positive_exclusion_union = unary_union(positive_buffers)

eligible_geom = malaysia_geom_m.difference(
    positive_exclusion_union
)

if eligible_geom.is_empty:
    raise ValueError(
        "No eligible sampling area remains after positive buffers."
    )

malaysia_area_km2 = malaysia_geom_m.area / 1_000_000
eligible_area_km2 = eligible_geom.area / 1_000_000
excluded_area_km2 = malaysia_area_km2 - eligible_area_km2

print(f"Malaysia boundary area in sampling CRS: {malaysia_area_km2:,.1f} km²")
print(f"Eligible pseudo-absence area: {eligible_area_km2:,.1f} km²")
print(f"Excluded positive-buffer area: {excluded_area_km2:,.1f} km²")
print(f"Eligible share: {eligible_area_km2 / malaysia_area_km2 * 100:.2f}%")

if eligible_area_km2 / malaysia_area_km2 < 0.50:
    print(
        "WARNING: Less than 50% of Malaysia remains eligible. "
        "Review large COOLR accuracy buffers before proceeding."
    )


Malaysia boundary area in sampling CRS: 329,819.9 km²
Eligible pseudo-absence area: 155,236.4 km²
Excluded positive-buffer area: 174,583.5 km²
Eligible share: 47.07%


## 6. Create a 50-km equal-area sampling grid

Pseudo-absence allocation will be proportional to **eligible land area per grid cell**.

This reduces the risk that random sampling becomes concentrated in the largest continuous landmass.


In [33]:
minx, miny, maxx, maxy = malaysia_geom_m.bounds

grid_cells = []
grid_ids = []

cell_id = 1

x = math.floor(minx / GRID_SIZE_M) * GRID_SIZE_M

while x < maxx:
    y = math.floor(miny / GRID_SIZE_M) * GRID_SIZE_M

    while y < maxy:
        cell = box(
            x,
            y,
            x + GRID_SIZE_M,
            y + GRID_SIZE_M
        )

        intersection = cell.intersection(eligible_geom)

        if not intersection.is_empty and intersection.area > 0:
            grid_cells.append(intersection)
            grid_ids.append(f"GRID_{cell_id:04d}")
            cell_id += 1

        y += GRID_SIZE_M

    x += GRID_SIZE_M

grid = gpd.GeoDataFrame(
    {
        "grid_id": grid_ids,
        "eligible_geometry": grid_cells,
    },
    geometry="eligible_geometry",
    crs=SAMPLING_CRS,
)

grid["eligible_area_m2"] = grid.geometry.area
grid["eligible_area_km2"] = grid["eligible_area_m2"] / 1_000_000

print("Eligible sampling grid cells:", len(grid))
print("Total grid eligible area:", grid["eligible_area_km2"].sum(), "km²")


Eligible sampling grid cells: 156
Total grid eligible area: 155236.37603853963 km²


## 7. Allocate the 1:3 master pseudo-absence sample across grid cells

Allocation is proportional to eligible area.

The largest-remainder method ensures the final allocated count equals exactly the required master count.


In [34]:
def proportional_integer_allocation(weights, total_n):
    weights = np.asarray(weights, dtype=float)

    if weights.sum() <= 0:
        raise ValueError("Allocation weights sum to zero.")

    raw = weights / weights.sum() * total_n
    base = np.floor(raw).astype(int)

    remainder = total_n - base.sum()

    if remainder > 0:
        fractional = raw - base
        add_idx = np.argsort(-fractional)[:remainder]
        base[add_idx] += 1

    return base


grid["target_master_n"] = proportional_integer_allocation(
    grid["eligible_area_m2"].to_numpy(),
    N_1TO3
)

print("Allocated master pseudo-absences:", grid["target_master_n"].sum())
print("Cells receiving >=1 point:", (grid["target_master_n"] > 0).sum())

display(
    grid[["grid_id", "eligible_area_km2", "target_master_n"]]
    .sort_values("target_master_n", ascending=False)
    .head(20)
)


Allocated master pseudo-absences: 861
Cells receiving >=1 point: 128


,grid_id,eligible_area_km2,target_master_n
56,GRID_0057,2489.816104,14
51,GRID_0052,2500.000000,14
49,GRID_0050,2499.014375,14
97,GRID_0098,2500.000000,14
99,GRID_0100,2447.741826,14
62,GRID_0063,2452.372023,14
61,GRID_0062,2500.000000,14
50,GRID_0051,2487.453806,14
52,GRID_0053,2500.000000,14
15,GRID_0016,2429.181624,13


## 8. Sampling functions

Points must satisfy:

- inside the eligible geometry,
- minimum separation from previously accepted pseudo-absence points.

A high attempt limit is used so the notebook fails explicitly instead of silently producing too few points.


In [35]:
# ============================================================
# 8. GENERATE BACKGROUND / PSEUDO-ABSENCE CANDIDATES
# ============================================================

def random_point_in_polygon(polygon, rng):

    minx, miny, maxx, maxy = polygon.bounds

    for _ in range(10000):

        x = rng.uniform(
            minx,
            maxx
        )

        y = rng.uniform(
            miny,
            maxy
        )

        candidate = Point(
            x,
            y
        )

        if polygon.covers(candidate):
            return candidate

    return None


def far_enough(
    candidate,
    existing_points,
    minimum_distance
):

    if len(existing_points) == 0:
        return True

    for existing in existing_points:

        if candidate.distance(existing) < minimum_distance:
            return False

    return True


# ------------------------------------------------------------
# Confirm grid geometry setup
# ------------------------------------------------------------

print(
    "Active grid geometry column:",
    grid.geometry.name
)

print(
    "Grid CRS:",
    grid.crs
)

print(
    "Number of eligible grid cells:",
    len(grid)
)


# ------------------------------------------------------------
# Generate points
# ------------------------------------------------------------

accepted_points = []
accepted_rows = []

MAX_ATTEMPTS_PER_CELL = 50000


for _, row in grid.iterrows():

    target_n = int(
        row["target_master_n"]
    )

    if target_n <= 0:
        continue

    # IMPORTANT:
    # Do not hard-code "geometry"
    cell_geom = row[
        grid.geometry.name
    ]

    if (
        cell_geom is None
        or cell_geom.is_empty
    ):
        continue

    accepted_in_cell = 0
    attempts = 0


    while (
        accepted_in_cell < target_n
        and attempts < MAX_ATTEMPTS_PER_CELL
    ):

        attempts += 1

        candidate = random_point_in_polygon(
            cell_geom,
            rng
        )

        if candidate is None:
            continue


        # ----------------------------------------
        # Pseudo-to-pseudo minimum separation
        # ----------------------------------------

        if not far_enough(
            candidate,
            accepted_points,
            MIN_PSEUDO_SEPARATION_M
        ):
            continue


        # ----------------------------------------
        # Accept candidate
        # ----------------------------------------

        accepted_points.append(
            candidate
        )

        accepted_rows.append({
            "pseudo_id":
                f"PA_{len(accepted_points):04d}",

            "grid_id":
                row["grid_id"],

            "sampling_seed":
                RANDOM_SEED,

            "min_pseudo_separation_m":
                MIN_PSEUDO_SEPARATION_M,

            "geometry":
                candidate
        })

        accepted_in_cell += 1


    # --------------------------------------------
    # Warning if target not reached
    # --------------------------------------------

    if accepted_in_cell < target_n:

        print(
            f"WARNING: "
            f"{row['grid_id']} requested "
            f"{target_n}, but generated "
            f"{accepted_in_cell} after "
            f"{attempts} attempts."
        )


# ------------------------------------------------------------
# Convert to GeoDataFrame
# ------------------------------------------------------------

pseudo_master_m = gpd.GeoDataFrame(
    accepted_rows,
    geometry="geometry",
    crs=SAMPLING_CRS
)


print()
print("-----------------------------------")
print("BACKGROUND GENERATION RESULT")
print("-----------------------------------")

print(
    "Generated:",
    len(pseudo_master_m)
)

print(
    "Required:",
    N_1TO3
)


if len(pseudo_master_m) != N_1TO3:

    raise RuntimeError(
        f"Generated {len(pseudo_master_m)} "
        f"points but expected {N_1TO3}."
    )

else:

    print()
    print(
        "SUCCESS: Required background "
        "candidate count generated."
    )

Active grid geometry column: eligible_geometry
Grid CRS: EPSG:6933
Number of eligible grid cells: 156

-----------------------------------
BACKGROUND GENERATION RESULT
-----------------------------------
Generated: 861
Required: 861

SUCCESS: Required background candidate count generated.


## 9. Validate pseudo-absence distances from positive exclusion zones


In [36]:
# All pseudo-absence points must be inside the eligible area.
pseudo_master_m["inside_eligible_area"] = (
    pseudo_master_m.geometry.apply(eligible_geom.covers)
)

print(
    "Inside eligible area:",
    pseudo_master_m["inside_eligible_area"].sum(),
    "/",
    len(pseudo_master_m)
)

if not pseudo_master_m["inside_eligible_area"].all():
    raise ValueError(
        "At least one pseudo-absence lies outside the eligible sampling area."
    )

# Direct minimum distance to any positive point.
positive_union_points = positive_m.geometry.tolist()

def minimum_positive_distance(point):
    return min(
        point.distance(p)
        for p in positive_union_points
    )

pseudo_master_m["nearest_positive_distance_m"] = (
    pseudo_master_m.geometry.apply(
        minimum_positive_distance
    )
)

print(
    "Minimum pseudo-absence → positive point distance:",
    round(
        pseudo_master_m["nearest_positive_distance_m"].min(),
        2
    ),
    "m"
)

# Because some positive exclusion buffers are larger than 2 km,
# the true eligibility check is the polygon check above.


Inside eligible area: 861 / 861
Minimum pseudo-absence → positive point distance: 2077.06 m


## 10. Validate pseudo-absence to pseudo-absence separation


In [37]:
coords = np.array(
    [
        [p.x, p.y]
        for p in pseudo_master_m.geometry
    ]
)

min_pair_distance = np.inf

for i in range(len(coords)):
    dxy = coords[i + 1:] - coords[i]

    if len(dxy) == 0:
        continue

    d = np.sqrt(
        (dxy ** 2).sum(axis=1)
    )

    local_min = d.min()

    if local_min < min_pair_distance:
        min_pair_distance = local_min

print(
    "Minimum pseudo↔pseudo separation:",
    round(min_pair_distance, 2),
    "m"
)

if min_pair_distance + 1e-6 < MIN_PSEUDO_SEPARATION_M:
    raise ValueError(
        "Pseudo-absence minimum separation check failed."
    )

print("Pseudo-absence spacing validation passed.")


Minimum pseudo↔pseudo separation: 2071.74 m
Pseudo-absence spacing validation passed.


## 11. Create nested 1:1, 1:2 and 1:3 sets

A fixed random ranking is assigned once.

This means:

- 1:1 is a subset of 1:2
- 1:2 is a subset of 1:3

So later model comparisons are easier to interpret.


In [38]:
pseudo_master_m = pseudo_master_m.copy()

rank_order = rng.permutation(
    len(pseudo_master_m)
)

pseudo_master_m["sampling_rank"] = np.empty(
    len(pseudo_master_m),
    dtype=int
)

pseudo_master_m.loc[
    rank_order,
    "sampling_rank"
] = np.arange(
    1,
    len(pseudo_master_m) + 1
)

pseudo_1to1_m = pseudo_master_m[
    pseudo_master_m["sampling_rank"] <= N_1TO1
].copy()

pseudo_1to2_m = pseudo_master_m[
    pseudo_master_m["sampling_rank"] <= N_1TO2
].copy()

pseudo_1to3_m = pseudo_master_m.copy()

print("1:1 pseudo count:", len(pseudo_1to1_m))
print("1:2 pseudo count:", len(pseudo_1to2_m))
print("1:3 pseudo count:", len(pseudo_1to3_m))


1:1 pseudo count: 287
1:2 pseudo count: 574
1:3 pseudo count: 861


## 12. Convert pseudo-absence master inventory to WGS84


In [39]:
pseudo_master = pseudo_master_m.to_crs("EPSG:4326")

pseudo_master["longitude"] = pseudo_master.geometry.x
pseudo_master["latitude"] = pseudo_master.geometry.y
pseudo_master["target"] = 0
pseudo_master["sample_class"] = "pseudo_absence"

pseudo_1to1 = pseudo_master[
    pseudo_master["sampling_rank"] <= N_1TO1
].copy()

pseudo_1to2 = pseudo_master[
    pseudo_master["sampling_rank"] <= N_1TO2
].copy()

pseudo_1to3 = pseudo_master.copy()

display(pseudo_master.head())


,pseudo_id,grid_id,sampling_seed,min_pseudo_separation_m,geometry,inside_eligible_area,nearest_positive_distance_m,sampling_rank,longitude,latitude,target,sample_class
0,PA_0001,GRID_0001,42,2000,POINT (99.85213 6.25585),True,46936.634155,49,99.852134,6.255855,0,pseudo_absence
1,PA_0002,GRID_0002,42,2000,POINT (99.67979 6.37066),True,59401.373804,460,99.679791,6.370659,0,pseudo_absence
2,PA_0003,GRID_0002,42,2000,POINT (99.8371 6.44302),True,44160.591856,860,99.837096,6.443016,0,pseudo_absence
3,PA_0004,GRID_0004,42,2000,POINT (100.50271 5.01438),True,32523.181469,500,100.502707,5.014383,0,pseudo_absence
4,PA_0005,GRID_0004,42,2000,POINT (100.49072 4.94872),True,28900.743162,529,100.490716,4.948716,0,pseudo_absence


## 13. Build combined modelling sample datasets

Positive records receive:

- `target = 1`
- `sample_class = landslide`

Pseudo-absence records receive:

- `target = 0`
- `sample_class = pseudo_absence`

These are **sample-location datasets only**.  
Environmental predictors will be added in the next feature-extraction stage.


In [40]:
positive_samples = positive.copy()

positive_samples["target"] = 1
positive_samples["sample_class"] = "landslide"
positive_samples["sample_id"] = positive_samples["final_positive_id"]

# Pseudo-absence standard fields.
def standardise_pseudo(df):
    out = df.copy()
    out["sample_id"] = out["pseudo_id"]
    out["source_dataset"] = "PSEUDO_ABSENCE"
    out["final_positive_id"] = pd.NA
    out["event_date"] = pd.NaT
    out["event_date_available"] = False
    return out


pseudo_1to1_std = standardise_pseudo(pseudo_1to1)
pseudo_1to2_std = standardise_pseudo(pseudo_1to2)
pseudo_1to3_std = standardise_pseudo(pseudo_1to3)

positive_keep_cols = [
    "sample_id",
    "target",
    "sample_class",
    "source_dataset",
    "final_positive_id",
    "event_date",
    "event_date_available",
    "longitude",
    "latitude",
    "geometry",
]

for col in positive_keep_cols:
    if col not in positive_samples.columns:
        positive_samples[col] = pd.NA

pseudo_keep_cols = positive_keep_cols

def combine_samples(pseudo_df):
    p = positive_samples[positive_keep_cols].copy()
    a = pseudo_df[pseudo_keep_cols].copy()

    return gpd.GeoDataFrame(
        pd.concat([p, a], ignore_index=True),
        geometry="geometry",
        crs="EPSG:4326"
    )


samples_1to1 = combine_samples(pseudo_1to1_std)
samples_1to2 = combine_samples(pseudo_1to2_std)
samples_1to3 = combine_samples(pseudo_1to3_std)

for name, df in {
    "1:1": samples_1to1,
    "1:2": samples_1to2,
    "1:3": samples_1to3,
}.items():
    print(
        name,
        "total =", len(df),
        "| positive =", int((df["target"] == 1).sum()),
        "| pseudo =", int((df["target"] == 0).sum())
    )


1:1 total = 574 | positive = 287 | pseudo = 287
1:2 total = 861 | positive = 287 | pseudo = 574
1:3 total = 1148 | positive = 287 | pseudo = 861


## 14. Baseline 1:2 spatial distribution summary


In [41]:
baseline_grid_counts = (
    pseudo_1to2["grid_id"]
    .value_counts()
    .rename_axis("grid_id")
    .reset_index(name="pseudo_absence_count")
)

display(
    baseline_grid_counts
    .sort_values(
        "pseudo_absence_count",
        ascending=False
    )
    .head(20)
)

baseline_grid_counts.to_csv(
    OUTPUT_DIR / "baseline_1to2_grid_distribution.csv",
    index=False
)


,grid_id,pseudo_absence_count
0,GRID_0058,12
1,GRID_0072,12
2,GRID_0052,11
3,GRID_0057,11
4,GRID_0022,10
5,GRID_0047,10
6,GRID_0051,10
7,GRID_0053,10
8,GRID_0062,10
9,GRID_0115,10


## 15. Map the baseline 1:2 sample design

This is a sampling-design map, not a susceptibility map.


In [42]:
malaysia_4326 = malaysia.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(12, 9))

malaysia_4326.boundary.plot(
    ax=ax,
    linewidth=0.8
)

pseudo_1to2.plot(
    ax=ax,
    markersize=10,
    alpha=0.5,
    label="Pseudo-absence (1:2 baseline)"
)

positive.plot(
    ax=ax,
    markersize=28,
    alpha=0.8,
    marker="x",
    label="Landslide positive"
)

ax.set_title(
    "Malaysia Positive and Pseudo-Absence Sampling Design (1:2 Baseline)"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

map_path = (
    OUTPUT_DIR
    / "baseline_1to2_sampling_map.png"
)

fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)

print("Saved:")
print(map_path)


Saved:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\pseudo_absence_generation_outputs\baseline_1to2_sampling_map.png


## 16. Save pseudo-absence master and nested datasets


In [43]:
# ------------------------------------------------------------
# Master pseudo-absence inventory
# ------------------------------------------------------------
pseudo_master_path = (
    CLEAN_DIR
    / "pseudo_absence_master_1to3.geojson"
)

pseudo_master.to_file(
    pseudo_master_path,
    driver="GeoJSON"
)

pseudo_master.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    CLEAN_DIR / "pseudo_absence_master_1to3.csv",
    index=False
)

# ------------------------------------------------------------
# Combined sample datasets
# ------------------------------------------------------------
dataset_map = {
    "1to1": samples_1to1,
    "1to2": samples_1to2,
    "1to3": samples_1to3,
}

for suffix, gdf in dataset_map.items():
    geojson_path = (
        CLEAN_DIR
        / f"modelling_samples_{suffix}.geojson"
    )

    csv_path = (
        CLEAN_DIR
        / f"modelling_samples_{suffix}.csv"
    )

    export = gdf.copy()

    if "event_date" in export.columns:
        export["event_date"] = (
            export["event_date"]
            .astype("string")
            .replace("<NA>", None)
        )

    export.to_file(
        geojson_path,
        driver="GeoJSON"
    )

    csv_export = export.drop(
        columns="geometry",
        errors="ignore"
    )

    csv_export.to_csv(
        csv_path,
        index=False
    )

    print("Saved:", geojson_path.name)
    print("Saved:", csv_path.name)


Saved: modelling_samples_1to1.geojson
Saved: modelling_samples_1to1.csv
Saved: modelling_samples_1to2.geojson
Saved: modelling_samples_1to2.csv
Saved: modelling_samples_1to3.geojson
Saved: modelling_samples_1to3.csv


## 17. Stage-4 quality-control summary


In [44]:
summary = pd.DataFrame({
    "check": [
        "Positive landslide records",
        "Master pseudo-absence target",
        "Master pseudo-absences generated",
        "Baseline 1:2 pseudo-absences",
        "Baseline total modelling samples",
        "Pseudo-absences outside eligible area",
        "Minimum pseudo-to-pseudo separation m",
        "Sampling grid size m",
        "Random seed",
        "Eligible Malaysia area km2",
        "Positive-buffer excluded area km2",
    ],
    "value": [
        len(positive),
        N_1TO3,
        len(pseudo_master),
        len(pseudo_1to2),
        len(samples_1to2),
        int((~pseudo_master_m["inside_eligible_area"]).sum()),
        round(float(min_pair_distance), 2),
        GRID_SIZE_M,
        RANDOM_SEED,
        round(float(eligible_area_km2), 2),
        round(float(excluded_area_km2), 2),
    ]
})

display(summary)

summary.to_csv(
    OUTPUT_DIR / "pseudo_absence_generation_summary.csv",
    index=False
)

print("\nSTAGE-4 DECISION")
print("----------------")
print(
    "Baseline recommendation: use modelling_samples_1to2.geojson "
    "for the first feature-extraction/model workflow."
)
print()
print(
    "Keep the 1:1 and 1:3 datasets for later sensitivity testing."
)
print()
print(
    "Pseudo-absence points are background controls, "
    "not verified non-landslide observations."
)
print()
print(
    "NEXT: 05_static_feature_extraction.ipynb"
)


,check,value
0,Positive landslide records,287.00
1,Master pseudo-absence target,861.00
2,Master pseudo-absences generated,861.00
3,Baseline 1:2 pseudo-absences,574.00
4,Baseline total modelling samples,861.00
5,Pseudo-absences outside eligible area,0.00
6,Minimum pseudo-to-pseudo separation m,2071.74
7,Sampling grid size m,50000.00
8,Random seed,42.00
9,Eligible Malaysia area km2,155236.38



STAGE-4 DECISION
----------------
Baseline recommendation: use modelling_samples_1to2.geojson for the first feature-extraction/model workflow.

Keep the 1:1 and 1:3 datasets for later sensitivity testing.

Pseudo-absence points are background controls, not verified non-landslide observations.

NEXT: 05_static_feature_extraction.ipynb


# Stage-4 output structure

After successful execution:

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   └── pseudo_absence_generation_outputs/
│       ├── positive_exclusion_buffer_summary.csv
│       ├── baseline_1to2_grid_distribution.csv
│       ├── baseline_1to2_sampling_map.png
│       └── pseudo_absence_generation_summary.csv
│
└── 02_cleaned_inventory/
    ├── pseudo_absence_master_1to3.geojson
    ├── pseudo_absence_master_1to3.csv
    ├── modelling_samples_1to1.geojson
    ├── modelling_samples_1to1.csv
    ├── modelling_samples_1to2.geojson
    ├── modelling_samples_1to2.csv
    ├── modelling_samples_1to3.geojson
    └── modelling_samples_1to3.csv
```

---

# Important interpretation

The selected baseline dataset contains:

- **287 observed landslide locations**
- **574 pseudo-absence/background locations**

for a total of:

- **861 sample locations**

This is **not yet the model-ready feature table**.

The next stage will extract the minimum static predictor set:

1. elevation
2. slope
3. aspect
4. land cover
5. distance to roads
6. distance to waterways

Then, where coverage is valid:

7. soil
8. geology
9. distance to faults

Rainfall remains separate until event-date logic is evaluated.


In [45]:
# ============================================================
# CHECK COOLR LOCATION ACCURACY
# ============================================================

coolr_accuracy_check = positive.loc[
    positive["source_dataset"] == "NASA_COOLR",
    [
        "final_positive_id",
        "source_record_id",
        "event_date",
        "state_or_admin",
        "district_or_locality",
        "event_title_or_name",
        "location_accuracy_source",
        "longitude",
        "latitude",
        "exclusion_buffer_m"
    ]
].copy()

coolr_accuracy_check = coolr_accuracy_check.sort_values(
    "exclusion_buffer_m",
    ascending=False
)

display(coolr_accuracy_check.head(30))

,final_positive_id,source_record_id,event_date,state_or_admin,district_or_locality,event_title_or_name,location_accuracy_source,longitude,latitude,exclusion_buffer_m
185,POS_0186,COOLR_9317,2016-02-18 00:00:00,Sarawak,Southern Sarawak,Southern Sarawak,250km,114.385900,3.333400,250000
26,POS_0027,COOLR_3061,2011-01-27 00:00:00,Sabah,"Siew Ching primary school(?), Kampung Pancuran...","Siew Ching primary school(?), Kampung Pancuran...",50km,117.955500,4.972300,50000
11,POS_0012,COOLR_5781,2014-01-14 09:00:00,Sabah,Sabah,Sabah,50km,116.796800,5.420400,50000
95,POS_0096,COOLR_320,2007-10-19 00:00:00,Sabah,Ranau-Tamparuli Road,Ranau-Tamparuli Road,50km,116.445300,6.043000,50000
142,POS_0143,COOLR_3077,2011-01-31 15:00:00,Sabah,"Kampung Sundang Darat(?), near Sandakan, Sabah","Kampung Sundang Darat(?), near Sandakan, Sabah",50km,117.980200,5.751500,50000
169,POS_0170,COOLR_4209,2012-02-18 01:30:00,Sabah,"Kampung Terusan, Lahad Datu","Kampung Terusan, Lahad Datu",50km,118.404700,5.088200,50000
168,POS_0169,COOLR_4685,2012-12-26 03:00:00,Terengganu,road connecting Kampung Air Putih and Kampung ...,road connecting Kampung Air Putih and Kampung ...,50km,103.217400,4.266400,50000
126,POS_0127,COOLR_2557,2010-10-07 23:00:00,Sabah,"road between Kota Kinabalu and Ranaum, Sabah s...","road between Kota Kinabalu and Ranaum, Sabah s...",50km,116.388000,5.960600,50000
132,POS_0133,COOLR_11050,2017-03-18 00:00:00,Sarawak,"Kg Rian, Song district, Kapit Division, Sarawa...",Kg Rian Landslide in the Song District,50km,112.595116,1.760867,50000
112,POS_0113,COOLR_4885,2013-05-29 00:00:00,Sabah,"Jalan Merakau, Ranau","Jalan Merakau, Ranau",50km,116.692800,5.987200,50000


In [46]:
display(
    coolr_accuracy_check[
        coolr_accuracy_check["exclusion_buffer_m"] >= 25000
    ]
)

,final_positive_id,source_record_id,event_date,state_or_admin,district_or_locality,event_title_or_name,location_accuracy_source,longitude,latitude,exclusion_buffer_m
185,POS_0186,COOLR_9317,2016-02-18 00:00:00,Sarawak,Southern Sarawak,Southern Sarawak,250km,114.385900,3.333400,250000
26,POS_0027,COOLR_3061,2011-01-27 00:00:00,Sabah,"Siew Ching primary school(?), Kampung Pancuran...","Siew Ching primary school(?), Kampung Pancuran...",50km,117.955500,4.972300,50000
11,POS_0012,COOLR_5781,2014-01-14 09:00:00,Sabah,Sabah,Sabah,50km,116.796800,5.420400,50000
95,POS_0096,COOLR_320,2007-10-19 00:00:00,Sabah,Ranau-Tamparuli Road,Ranau-Tamparuli Road,50km,116.445300,6.043000,50000
142,POS_0143,COOLR_3077,2011-01-31 15:00:00,Sabah,"Kampung Sundang Darat(?), near Sandakan, Sabah","Kampung Sundang Darat(?), near Sandakan, Sabah",50km,117.980200,5.751500,50000
169,POS_0170,COOLR_4209,2012-02-18 01:30:00,Sabah,"Kampung Terusan, Lahad Datu","Kampung Terusan, Lahad Datu",50km,118.404700,5.088200,50000
168,POS_0169,COOLR_4685,2012-12-26 03:00:00,Terengganu,road connecting Kampung Air Putih and Kampung ...,road connecting Kampung Air Putih and Kampung ...,50km,103.217400,4.266400,50000
126,POS_0127,COOLR_2557,2010-10-07 23:00:00,Sabah,"road between Kota Kinabalu and Ranaum, Sabah s...","road between Kota Kinabalu and Ranaum, Sabah s...",50km,116.388000,5.960600,50000
132,POS_0133,COOLR_11050,2017-03-18 00:00:00,Sarawak,"Kg Rian, Song district, Kapit Division, Sarawa...",Kg Rian Landslide in the Song District,50km,112.595116,1.760867,50000
112,POS_0113,COOLR_4885,2013-05-29 00:00:00,Sabah,"Jalan Merakau, Ranau","Jalan Merakau, Ranau",50km,116.692800,5.987200,50000


In [47]:
coolr_accuracy_check[
    coolr_accuracy_check["exclusion_buffer_m"] >= 10000
].to_csv(
    OUTPUT_DIR / "coolr_low_precision_review.csv",
    index=False
)